# Notebook 01: EDA and Data Preparation

NHS England Monthly A&E Time Series — exploratory analysis, feature engineering, and train/validation/test split.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import save_master_dataset
from src.features import TARGET, add_period_flags, prepare_modeling_frame, split_dataframe

FIG_DIR = PROJECT_ROOT / 'outputs' / 'figures'
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')

In [ ]:
raw = save_master_dataset()
df = add_period_flags(raw)
model_df = prepare_modeling_frame(raw)
model_df.to_csv(TABLE_DIR / 'modeling_dataset.csv', index=False)

print(f'Observations: {len(df)}')
print(f'Period: {df.period.min().date()} to {df.period.max().date()}')
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df['period'], df[TARGET], linewidth=2, color='#1f4e79')
ax.axvspan('2019-05-01', '2023-05-01', alpha=0.15, color='orange', label='CRS period')
ax.axvspan('2020-03-01', '2021-06-01', alpha=0.15, color='red', label='COVID period')
ax.set_title('Monthly A&E Four-Hour Performance (England)')
ax.set_ylabel('% seen within 4 hours')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '01_performance_trend.png')
plt.show()

In [ ]:
tmp = df.dropna(subset=[TARGET]).copy()
tmp['month_name'] = tmp['period'].dt.month_name()
month_order = list(pd.date_range('2020-01-01', periods=12, freq='MS').month_name())

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=tmp, x='month_name', y=TARGET, order=month_order, ax=ax)
ax.set_title('Seasonal Distribution by Month')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / '02_seasonal_boxplot.png')
plt.show()

In [ ]:
summary = df[[TARGET, 'total_attendances', 'emergency_admissions']].describe().round(2)
summary.to_csv(TABLE_DIR / 'eda_summary_statistics.csv')
summary

In [ ]:
train, val, test = split_dataframe(model_df)
print('Train:', len(train), train.period.min().date(), 'to', train.period.max().date())
print('Validation:', len(val), val.period.min().date(), 'to', val.period.max().date())
print('Test:', len(test), test.period.min().date(), 'to', test.period.max().date())
model_df['split'].value_counts()